# PaySim Fraud Risk Analysis 2.0
## Phase 2 — Silver: Data Preparation

### Objective
Prepare and transform the transaction data for fraud risk analysis.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/PS_20174392719_1491204439457_log.csv")

In [3]:
df.head(5)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
df["high_risk_type_flag"] = df["type"].isin(["TRANSFER", "CASH_OUT"]).astype(int)

In [5]:
df[["type", "high_risk_type_flag"]].head(10)

,type,high_risk_type_flag
0,PAYMENT,0
1,PAYMENT,0
2,TRANSFER,1
3,CASH_OUT,1
4,PAYMENT,0
5,PAYMENT,0
6,PAYMENT,0
7,PAYMENT,0
8,PAYMENT,0
9,DEBIT,0


### High-Risk Transaction Classification

A new variable, `high_risk_type_flag`, was created to identify transactions belonging to the TRANSFER and CASH_OUT categories.

- **1:** TRANSFER or CASH_OUT transaction.
- **0:** All other transaction types.

These categories were selected because the Bronze analysis identified recorded fraudulent transactions exclusively within them.

**Important:** This flag identifies transaction categories associated with observed fraud, not confirmed fraudulent transactions.

In [6]:
df["high_risk_type_flag"].value_counts()

high_risk_type_flag
0    3592211
1    2770409
Name: count, dtype: int64

In [7]:
high_risk_percentage = df["high_risk_type_flag"].mean() * 100
print(high_risk_percentage)

43.54195284332555


### High-Risk Flag Distribution

The dataset contains 6,362,620 transactions.

- **2,770,409 transactions (43.54%)** were assigned a high-risk type flag of 1.
- **3,592,211 transactions (56.46%)** were assigned a flag of 0.

**Key Insight:** Nearly 44% of all transactions belong to categories associated with observed fraud. This represents a large potential investigation workload.

**Business Implication:** Transaction type alone may not provide enough precision for an efficient fraud investigation strategy. Additional indicators should be evaluated to improve prioritization.

In [8]:
df.groupby("high_risk_type_flag")["isFraud"].sum()

high_risk_type_flag
0       0
1    8213
Name: isFraud, dtype: int64

### Fraud Distribution by Risk Flag

The validation confirms that all 8,213 recorded fraudulent transactions belong to the group identified by `high_risk_type_flag = 1`.

No recorded fraudulent transactions were identified in the group with a flag of 0.

**Key Insight:** The indicator captures all known fraud cases in this dataset, but it also flags a substantial number of legitimate transactions.

**Business Implication:** Although the indicator provides complete coverage of recorded fraud, its effectiveness as a standalone investigation rule is limited by the large number of transactions selected for review.

**Next Step:** Evaluate the fraud rate within the flagged group and explore additional transaction characteristics to improve risk prioritization.

In [9]:
flag_precision = (8213 / 2770409) * 100
print(flag_precision)

0.2964544224336551


In [10]:
## Calculate flag precision without hardcoding

total_frauds = df["isFraud"].sum()
total_flagged = df["high_risk_type_flag"].sum()

flag_precision = (total_frauds / total_flagged) * 100

print(f"Total frauds: {total_frauds}")
print(f"Total flagged transactions: {total_flagged}")
print(f"Flag precision: {flag_precision:.2f}%")

Total frauds: 8213
Total flagged transactions: 2770409
Flag precision: 0.30%


### Flag Precision — Key Findings

The high-risk transaction flag identified 2,770,409 transactions, including 8,213 confirmed fraudulent transactions.

**Flag Precision: 0.2965%**

**Key Findings:**
- Approximately 3 out of every 1,000 flagged transactions were fraudulent.
- More than 99.7% of flagged transactions were legitimate.
- The flag captures all recorded fraud cases but has very low precision.

**Business Implications:**

Using transaction type alone to prioritize fraud investigations would create a substantial workload with many false positives.

Additional transaction characteristics should be evaluated to improve precision while maintaining strong fraud detection coverage.

**Next Step:** Explore additional risk indicators to identify a smaller, more targeted group of transactions for investigation.

In [11]:
df.groupby("isFraud")["amount"].mean()

isFraud
0    1.781970e+05
1    1.467967e+06
Name: amount, dtype: float64

### Transaction Amount — Key Findings

The analysis compared average transaction amounts between legitimate and fraudulent transactions.

**Results:**
- Average legitimate transaction amount: 178,197
- Average fraudulent transaction amount: 1,467,967
- Fraudulent transactions had an average amount approximately 8.2 times higher than legitimate transactions.

**Key Insight:** Transaction amount may be a useful indicator for identifying potentially fraudulent activity.

**Business Implication:** Incorporating transaction amount into the risk assessment process could help prioritize transactions for further investigation.

**Next Step:** Examine the median transaction amount to determine whether unusually large transactions are influencing the averages.

In [12]:
df.groupby("isFraud")["amount"].median()

isFraud
0     74684.72
1    441423.44
Name: amount, dtype: float64

### Transaction Amount — Mean vs. Median

**Key Findings:**

- The median fraudulent transaction amount was 441,423.44, compared with 74,684.72 for legitimate transactions.
- The median fraudulent transaction was approximately 5.9 times larger than the median legitimate transaction.
- Both groups showed mean values substantially higher than their medians, suggesting right-skewed distributions influenced by large transaction amounts.

**Business Implication:**

Transaction amount may help improve fraud risk prioritization. However, additional analysis is needed before selecting an amount-based risk threshold.

**Next Step:** Examine transaction amount distributions and identify potential outliers.

In [13]:
df.groupby("isFraud")["amount"].max()

isFraud
0    92445516.64
1    10000000.00
Name: amount, dtype: float64

### Transaction Amount — Maximum Values

**Results:**
- Maximum legitimate transaction amount: 92,445,516.64
- Maximum fraudulent transaction amount: 10,000,000.00

**Key Findings:**

The largest legitimate transaction was more than nine times larger than the largest fraudulent transaction.

Although fraudulent transactions showed higher average and median amounts, extremely large transactions were also present among legitimate transactions.

**Business Implication:**

Transaction amount alone is not sufficient to identify fraud reliably. An amount-based risk rule could generate false positives and should be evaluated alongside additional transaction characteristics.

**Next Step:** Examine transaction amount distributions to better understand extreme values and potential risk thresholds.

In [14]:
df.groupby("isFraud")["amount"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,1.781970e+05,5.962370e+05,0.01,13368.395,74684.72,208364.76,92445516.64
1,8213.0,1.467967e+06,2.404253e+06,0.00,127091.330,441423.44,1517771.48,10000000.00


### Transaction Amount Distribution — Key Findings

**Objective:** Compare transaction amount distributions between legitimate and fraudulent transactions using quartiles.

**Key Findings:**

- The median fraudulent transaction amount was 441,423.44, compared with 74,684.72 for legitimate transactions.
- The 25th percentile was 127,091.33 for fraudulent transactions and 13,368.40 for legitimate transactions.
- The 75th percentile was 1,517,771.48 for fraudulent transactions and 208,364.76 for legitimate transactions.
- Fraudulent transactions generally involved larger amounts, although the amount distributions overlap.

**Business Implication:**

Transaction amount is a promising risk indicator, but it should be combined with other transaction characteristics rather than used as a standalone fraud classification rule.

**Next Step:** Explore potential transaction amount thresholds and evaluate their impact on fraud detection and false positives.

In [15]:
df["amount"] > 100000

0          False
1          False
2          False
3          False
4          False
           ...  
6362615     True
6362616     True
6362617     True
6362618     True
6362619     True
Name: amount, Length: 6362620, dtype: bool

In [16]:
df["amount_threshold_flag"] = (df["amount"] > 100000).astype(int)

In [17]:
df[["amount", "amount_threshold_flag"]].head(10)

,amount,amount_threshold_flag
0,9839.64,0
1,1864.28,0
2,181.00,0
3,181.00,0
4,11668.14,0
5,7817.71,0
6,7107.77,0
7,7861.64,0
8,4024.36,0
9,5337.77,0


In [18]:
df["combined_risk_flag"] = (
    (df["high_risk_type_flag"] == 1) &
    (df["amount_threshold_flag"] == 1)
).astype(int)

In [19]:
df["combined_risk_flag"].value_counts()

combined_risk_flag
0    4428323
1    1934297
Name: count, dtype: int64

In [20]:
df.groupby("combined_risk_flag")["isFraud"].sum()

combined_risk_flag
0    1707
1    6506
Name: isFraud, dtype: int64

In [21]:
recall = (6506 / 8213) * 100
print(recall)

79.21587726774627


### Combined Risk Flag — Precision and Recall Analysis

**Objective:** Evaluate whether combining transaction type with an amount threshold of 100,000 improves fraud investigation efficiency.

**Key Findings:**

- The original transaction-type flag selected 2,770,409 transactions and captured all 8,213 recorded fraud cases.
- The combined rule selected 1,934,297 transactions, reducing the investigation volume by 30.18%.
- Precision increased slightly from approximately 0.30% to 0.34%.
- Recall decreased from 100% to 79.22%, leaving 1,707 fraudulent transactions undetected.

**Business Implications:**

The combined rule reduces the number of transactions requiring investigation but sacrifices fraud detection coverage.

The modest improvement in precision may not justify missing approximately 21% of recorded fraud cases.

**Conclusion:**

The 100,000 amount threshold should not be adopted without further evaluation. Additional risk indicators and alternative thresholds should be explored to improve precision while maintaining stronger recall.

In [22]:
df[[
    "high_risk_type_flag",
    "amount_threshold_flag",
    "combined_risk_flag"
]].isnull().sum()

high_risk_type_flag      0
amount_threshold_flag    0
combined_risk_flag       0
dtype: int64

In [23]:
for column in [
    "high_risk_type_flag",
    "amount_threshold_flag",
    "combined_risk_flag"
]:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))


high_risk_type_flag
high_risk_type_flag
0    3592211
1    2770409
Name: count, dtype: int64

amount_threshold_flag
amount_threshold_flag
0    3525298
1    2837322
Name: count, dtype: int64

combined_risk_flag
combined_risk_flag
0    4428323
1    1934297
Name: count, dtype: int64


### Silver Data Validation — Key Findings

**Objective:** Validate the completeness and allowed values of the three newly created risk indicators.

**Validation Results:**

- All three risk indicator columns contain zero missing values.
- Each indicator contains only binary values (0 and 1).
- The value counts for each indicator sum to 6,362,620, matching the total number of transactions.

**Conclusion:**

The three risk indicators passed the completeness and binary-value validation checks.

A final consistency check will verify that the indicators accurately reflect their intended classification rules.

In [24]:
df.groupby([
    "high_risk_type_flag",
    "amount_threshold_flag",
    "combined_risk_flag"
]).size()

high_risk_type_flag  amount_threshold_flag  combined_risk_flag
0                    0                      0                     2689186
                     1                      0                      903025
1                    0                      0                      836112
                     1                      1                     1934297
dtype: int64

### Final Risk Indicator Consistency Validation

**Objective:** Verify that the combined risk indicator correctly applies both transaction-type and amount-threshold conditions.

**Validation Results:**

- 1,934,297 transactions satisfied both conditions and received `combined_risk_flag = 1`.
- 4,428,323 transactions did not satisfy both conditions and received `combined_risk_flag = 0`.
- No inconsistent flag combinations were identified.
- All 6,362,620 transactions were accounted for.

**Conclusion:**

The combined risk indicator passed the consistency validation.

All three newly created indicators passed the missing-value, binary-value, and logical-consistency checks performed during the Silver phase.

**Next Step:** Finalize the prepared dataset and document its limitations before proceeding to the Gold analytical layer.

In [25]:
df.to_csv("../data/paysim_silver.csv", index=False)
print("CSV saved successfully!")

CSV saved successfully!


### Silver Phase — Final Summary and Limitations

**Objective:** Prepare and validate transaction-level risk indicators for further fraud analysis.

**Completed Work:**
- Created three binary risk indicators based on transaction type, amount, and combined conditions.
- Analyzed transaction amounts using descriptive statistics and percentiles.
- Evaluated fraud detection performance using precision and recall.
- Validated missing values, binary values, and logical consistency.
- Exported the prepared dataset as `paysim_silver.csv`.

**Key Findings:**

The original transaction-type flag captured all 8,213 recorded fraud cases but selected 2,770,409 transactions for investigation.

The combined rule reduced the investigation volume by 30.18%, while recall decreased to 79.22% and precision increased slightly to approximately 0.34%.

**Limitations:**

- The dataset is simulated and may not represent real-world banking behavior.
- The 100,000 amount threshold was selected for exploratory analysis, not optimized or validated for operational use.
- Both rules produce substantial numbers of false positives.
- The analysis uses known fraud labels from the same dataset; results have not been evaluated on independent data.
- The risk indicators identify transactions for potential review but do not establish whether a transaction is fraudulent.

**Next Phase — Gold:**

Develop risk analysis summaries, compare alternative investigation strategies, and create reporting-ready datasets and visualizations to communicate fraud detection performance and business trade-offs.